In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Trend, Seasonality, and Noise

This notebook aims to introduce the core components that define the structure of a time series:
- **Trend** - long-term direction in the data
- **Seasonality** - repeating patterns over fixed intervals
- **Noise** - random variation

It is generally accepted that real (i.e., observed) time series data will always contain some element of noise, for example due to instrument inaccuracy, environmental variability, or human error. The presence of trend and seasonality components, however, is domain-dependent. For instance, local temperature data typically exhibits both:
- trend: due to long-term climate change, urbanisation, etc., and
- seasonality: reflecting annual seasonal cycles (e.g., summers being warmer than winters).

Other domains may show neither. For example, a series representing measurement noise from a sensor under static conditions would ideally have no trend or seasonality, just white noise.

Therefore, when analysing a time series, it's critical to assess which structural components are present.

### Additive vs Multiplicative

Before we delve deeper into time series analysis, it is important to understand how the components combine. Two common approaches are the additive and multiplicative models:

- Additive model: The observed time series is the sum of the components:
  $$ y(t) = Trend(t) + Seasonality(t) + Noise(t)$$ 
    This model is appropriate when seasonal fluctuations remain roughly constant regardless of the overall level of the series.

- Multiplicative model: The components multiply together:
  $$ y(t) = Trend(t) \times Seasonality(t) \times Noise(t)$$ 
    Use this model when the size of seasonal variations changes proportionally with the series level (e.g., seasonal peaks grow as the trend increases).

An example illustraing the differences is shown below.

In [ ]:
n = 100
x = np.arange(n)

# Components
trend = np.linspace(1, 10, n)
seasonality = 1 + 2 * np.sin(2 * np.pi * x / 24)
noise_add = np.random.normal(0, 0.5, n)
noise_mul = 1 + np.random.normal(0, 0.3, n)

y_additive = trend + seasonality + noise_add
y_multiplicative = trend * seasonality * noise_mul

# Plot
fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
sns.lineplot(x=x, y=y_additive, ax=axs[0])
axs[0].set_title("Additive Model")
sns.lineplot(x=x, y=y_multiplicative, ax=axs[1])
axs[1].set_title("Multiplicative Model")

plt.tight_layout()

Observations:
- In the additive model, the seasonal component maintains a constant amplitude over time.
- In contrast, the multiplicative model assumes that the amplitude of the seasonal variation changes proportionally with the trend. As the trend increases or decreases, the seasonal peaks and troughs scale accordingly, resulting in larger fluctuations at higher trend levels and smaller ones at lower levels.

Further notes:
- It should be noted that real, observed data rarely conforms perfectly to either a purely additive or purely multiplicative model.
- In practice, many time series exhibit elements of both
- These mixed patterns highlight the importance of carefully assessing the nature of your data before applying a specific model.

In this notebook, we focus primarily on the additive model, which is simpler to understand and suitable for many real-world scenarios. 

## Components

We shall now examine the components in their isolation, with noise, and in their summation.

In [ ]:
# generate a years worth of daily data
n = 365
x_dates = pd.date_range(start=pd.Timestamp("2025-01-01"), periods=n, freq="1D")
x_dates[:10]

In [ ]:
def plot_single(x, y, title=""):
    fig, ax = plt.subplots()
    sns.lineplot(x=x, y=y, ax=ax)

    ax.set(title=title, xlabel="Date")
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))
    fig.tight_layout()

### Noise

- This plot shows pure noise, which has random variation with no predictable pattern
- Noise typically has a constant mean (often zero) and constant variance, and it represents the unpredictable part of the data
- Real-world measurements always contain some noise due to factors like instrument error or environmental randomness

In [ ]:
noise = np.random.normal(0, 1, n)
plot_single(x_dates, noise, "White noise")

### Trend

- This plot shows a trend, a long-term increase (or decrease) in the values over time
- Trend represents persistent changes that can be caused by external factors (e.g., improvement in performance, economic growth)
- Trend causes the average (mean) value to change over time

In [ ]:
y_trend = np.linspace(0, 10, n)
plot_single(x_dates, y_trend, "Linear trend")

#### Trend + Noise

- This plot combines the trend with noise, showing a more realistic scenario where the underlying signal is obscured by random fluctuations

In [ ]:
y_trend_noise = y_trend + noise
plot_single(x_dates, y_trend_noise, "Linear trend + noise")

### Seasonality

- This plot shows a repeating cyclical pattern, where values rise and fall in a regular, predictable way (e.g., daily, weekly, or yearly cycles)
- Seasonality often reflects natural or human-made cycles, such as weather seasons or weekly work patterns
- Over longer time spans, seasonality causes no or little change in average over time

In [ ]:
# period of 30 as thats ~monthly
period = 30
y_seasonal = 1 * np.sin(2 * np.pi * np.arange(n) / period)
plot_single(x_dates, y_seasonal, "Seasonality")

#### Seasonality + Noise

- This plot shows seasonal patterns combined with noise, simulating real-world data where cycles exist but are masked by random variation.

In [ ]:
y_seasonal_noise = y_seasonal + noise
plot_single(x_dates, y_seasonal_noise, "Seasonality + noise")

### Trend + Seasonality + Noise

- This plot shows the combination of trend, seasonality, and noise, representing a typical complex real-world time series
- The trend causes the overall upward movement, seasonality causes periodic ups and downs, and noise adds randomness

Understanding and separating these components is crucial for effective time series analysis and forecasting.

In [ ]:
y_all = y_trend + y_seasonal + noise
plot_single(x_dates, y_all, "Trend + Seasonality + Noise")

## Time Series Decomposition

Above, we have seen how the constituent components of a time series combine to create patterns that can both reveal and obscure meaningful information, complicating analysis. Let's now look at techniques for isolating and analysing the individual components of a time series. The following techniques are commonly used to separate a time series into its underlying components:
- Moving Averages
- Classical Decomposition
- Seasonal-Trend decomposition using Loess (STL)

For ease, lets create a `pandas.Series` from the above data. 

In [ ]:
series = pd.Series(y_all, index=x_dates)
series.head()

### Moving Averages

In [ ]:
window = 12
windows = series.rolling(window=window, center=True).mean()
series.plot(label="Original", alpha=0.5)
windows.plot(label=f"Moving average\n{window=}")

plt.title(f"Trend Estimation with Moving Average")
plt.legend()
plt.tight_layout()

In [ ]:
series.plot(label="Original", alpha=0.5)
for window in (10, 20, 30):
    series.rolling(window=window, center=True).mean().plot(label=f"{window=}")

plt.title("Varying window lenth with Moving Average")
plt.legend()
plt.tight_layout()

### Classical Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# period of 30 as thats ~monthly with which we created the data
decomp = seasonal_decompose(series, model='additive', period=period)

decomp.plot()
plt.suptitle("Classical Additive Decomposition")
plt.tight_layout()

### Seasonal-Trend decomposition using Loess (STL)

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(series, period=period)
result = stl.fit()

result.plot()
plt.suptitle("STL Decomposition")
plt.tight_layout()

### Comparison

We shall now compare the results of classical and STL decomposition side by side.

In [ ]:
# Classical 
classical_result = seasonal_decompose(series, model='additive', period=period)

# STL 
stl = STL(series, period=period)
stl_result = stl.fit()

fig, axs = plt.subplots(3, 2, figsize=(10, 5), sharex=True)

# Classical (LHS)
axs[0, 0].plot(classical_result.trend)
axs[0, 0].set_title('Classical')
axs[1, 0].plot(classical_result.seasonal)
axs[2, 0].plot(classical_result.resid)

# STL (RHS)
axs[0, 1].set_title('STL')
axs[0, 1].plot(stl_result.trend)
axs[1, 1].plot(stl_result.seasonal)
axs[2, 1].plot(stl_result.resid)

row_titles = ['Trend', 'Seasonal', 'Residual']
for i in range(3):
    axs[i, 0].set_ylabel(row_titles[i])

for ax in axs[2, :]:
    ax.set_xlabel('Time')

plt.tight_layout()

Observations:
- Trend: Both methods identify a similar long-term trend. However, the STL trend appears noticeably smoother, as it uses LOESS smoothing, whereas the classical method relies on moving averages.
- Seasonality: Classical decomposition assumes a fixed, repeating seasonal pattern and thus produces a highly consistent seasonal component. STL however allows the seasonal pattern to vary over time, which can lead to more flexible but less stable seasonal estimates.
- Residuals (Noise): The residuals from STL decomposition appear to have greater variance. This is likely due to STL allocating more variation to the residual component, especially when the seasonal or trend patterns shift over time.